## Setup

Imports and environment. `HF_HOME` redirects the Hugging Face model cache to scratch storage so 8B-parameter weights don't fill the home quota.

In [1]:
# Smoke-test switch: run a fast end-to-end pipeline with tiny data before committing to the full run.
#   True  -> 5 candidates, 2 few-shot, ~500-row train pool (~12 min total incl. training smoke)
#   False -> 200 candidates, 5 few-shot, full ~14k-row train pool (real run)
SMOKE_TEST = False

import os
os.environ["HF_HOME"] = "/rds/general/user/rmw324/home/raels_playground/hugging_face_llms"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import gc, re
import numpy as np
import pandas as pd
import torch

import dllm
from dllm.core.samplers import MDLMSampler, MDLMSamplerConfig

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print(f"SMOKE_TEST = {SMOKE_TEST}")


torch: 2.6.0+cu124 | cuda: True
SMOKE_TEST = False


## Load data

Read the PetFinder training table and the breed/colour lookup tables. `to_readable` swaps numeric codes (Gender=1, MaturitySize=2, …) for human-readable strings — needed because the language models expect English, not category IDs.

In [2]:
DATA_DIR = "/rds/general/user/rmw324/home/raels_playground/datasets/pet_finder"

train = pd.read_csv(os.path.join(DATA_DIR, "train/train.csv"))
breed_labels = pd.read_csv(os.path.join(DATA_DIR, "BreedLabels.csv"))
color_labels = pd.read_csv(os.path.join(DATA_DIR, "ColorLabels.csv"))

breed_map = dict(zip(breed_labels["BreedID"], breed_labels["BreedName"]))
color_map = dict(zip(color_labels["ColorID"], color_labels["ColorName"]))

GENDER_MAP = {1: "Male", 2: "Female", 3: "Mixed"}
SIZE_MAP   = {1: "Small", 2: "Medium", 3: "Large", 4: "Extra Large"}
FUR_MAP    = {1: "Short", 2: "Medium", 3: "Long"}
YESNO_MAP  = {1: "Yes", 2: "No", 3: "Not Sure"}
HEALTH_MAP = {1: "Healthy", 2: "Minor Injury", 3: "Serious Injury"}
TYPE_MAP   = {1: "Dog", 2: "Cat"}

def to_readable(df):
    out = df.copy()
    out["Type"]         = out["Type"].map(TYPE_MAP)
    out["Gender"]       = out["Gender"].map(GENDER_MAP)
    out["MaturitySize"] = out["MaturitySize"].map(SIZE_MAP)
    out["FurLength"]    = out["FurLength"].map(FUR_MAP)
    out["Vaccinated"]   = out["Vaccinated"].map(YESNO_MAP)
    out["Dewormed"]     = out["Dewormed"].map(YESNO_MAP)
    out["Sterilized"]   = out["Sterilized"].map(YESNO_MAP)
    out["Health"]       = out["Health"].map(HEALTH_MAP)
    out["Breed1"]       = out["Breed1"].map(breed_map).fillna("Unknown")
    out["Breed2"]       = out["Breed2"].map(breed_map).fillna("None")
    out["Color1"]       = out["Color1"].map(color_map).fillna("Unknown")
    out["Color2"]       = out["Color2"].map(color_map).fillna("None")
    out["Color3"]       = out["Color3"].map(color_map).fillna("None")
    return out

print(train.shape)
train.head(3)

(14993, 24)


,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed
0,2,Nibble,3,299,0,1,1,7,0,1,...,1,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,86e1089a3,1.0,2
1,2,No Name Yet,1,265,0,1,1,2,0,2,...,1,1,0,41401,3082c7125d8fb66f7dd4bff4192c8b14,0,I just found it alone yesterday near my apartm...,6296e909a,2.0,0
2,1,Brisco,1,307,0,1,2,7,0,2,...,1,1,0,41326,fa90fa5b1ee11c86938398b60abc32cb,0,Their pregnant mother was dumped by her irresp...,3422e4906,7.0,3


In [3]:
train["Description"].str.len().describe()

count    14980.000000
mean       339.607543
std        373.393490
min          1.000000
25%        117.000000
50%        238.000000
75%        432.000000
max       6664.000000
Name: Description, dtype: float64

## Pick descriptive rows

Keep 25 rows whose `Description` is long enough and contains words that hint at hidden fields ("he/she", colours, sizes, "vaccinated", …). These cues give the language models something to actually reason from when fields are masked.

In [4]:
# --- Candidate selection (cue-filtered rows held out from training) ---
N_CANDIDATES = 5 if SMOKE_TEST else 200

CUE = re.compile(
    r"\b(?:he|she|him|her|his|hers|male|female|boy|girl|"
    r"black|white|brown|gray|grey|orange|yellow|tabby|cream|golden|"
    r"small|medium|large|tiny|huge|big|"
    r"short|long|fluffy|"
    r"vaccinated|sterilized|spayed|neutered|dewormed|kitten|puppy)\b",
    re.IGNORECASE,
)

mask = (
    train["Description"].notna()
    & (train["Description"].str.len() > 200)
    & train["Description"].str.contains(CUE, na=False, regex=True)
)
all_matching = train[mask]
print(f"Cue filter matched {len(all_matching)} rows; taking up to {N_CANDIDATES}")
candidates = all_matching.head(N_CANDIDATES).reset_index(drop=True)
candidates_readable = to_readable(candidates)
# Original train indices (for excluding from train_pool later)
candidate_train_indices = train.index[mask].tolist()[:len(candidates_readable)]
print(f"Selected {len(candidates_readable)} candidate rows")
candidates_readable.head()


Cue filter matched 7618 rows; taking up to 200
Selected 200 candidate rows


,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PetID,PhotoAmt,AdoptionSpeed
0,Cat,Nibble,3,Tabby,None,Male,Black,White,None,Small,...,Healthy,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,86e1089a3,1.0,2
1,Dog,Brisco,1,Mixed Breed,None,Male,Brown,White,None,Medium,...,Healthy,1,0,41326,fa90fa5b1ee11c86938398b60abc32cb,0,Their pregnant mother was dumped by her irresp...,3422e4906,7.0,3
2,Dog,Hunter,1,Mixed Breed,None,Male,Black,None,None,Medium,...,Healthy,1,0,41326,95481e953f8aed9ec3d16fc4509537e8,0,This handsome yet cute boy is up for adoption....,850a43f90,3.0,2
3,Cat,BULAT,12,Domestic Long Hair,Domestic Long Hair,Male,Black,None,None,Medium,...,Healthy,1,300,41326,1e0b5a458b5b77f5af581d57ebf570b3,0,anyone within the area of ipoh or taiping who ...,1caa6fcdb,3.0,1
4,Cat,Kitty,12,Domestic Medium Hair,None,Female,Black,White,None,Medium,...,Healthy,1,0,41326,1f3f36e4b18e94855b3e88af0852fdc4,0,"Very manja and gentle stray cat found, we woul...",7a0942d61,2.0,4


## Inspect candidates

Print every selected row in full so we can manually decide which cells to hide in the next step.

In [5]:
DISPLAY_FIELDS = ["Type","Name","Age","Breed1","Gender","Color1",
                  "MaturitySize","FurLength","Vaccinated","Sterilized",
                  "Health","Description"]

# With 200 candidates, only print a few for sanity
for i, row in candidates_readable.head(3).iterrows():
    print(f"\n=== Row {i} ===")
    for f in DISPLAY_FIELDS:
        val = str(row[f])
        if f == "Description" and len(val) > 500:
            val = val[:500] + "..."
        print(f"  {f}: {val}")
print(f"\n... ({len(candidates_readable)} candidates total)")



=== Row 0 ===
  Type: Cat
  Name: Nibble
  Age: 3
  Breed1: Tabby
  Gender: Male
  Color1: Black
  MaturitySize: Small
  FurLength: Short
  Vaccinated: No
  Sterilized: No
  Health: Healthy
  Description: Nibble is a 3+ month old ball of cuteness. He is energetic and playful. I rescued a couple of cats a few months ago but could not get them neutered in time as the clinic was fully scheduled. The result was this little kitty. I do not have enough space and funds to care for more cats in my household. Looking for responsible people to take over Nibble's care.

=== Row 1 ===
  Type: Dog
  Name: Brisco
  Age: 1
  Breed1: Mixed Breed
  Gender: Male
  Color1: Brown
  MaturitySize: Medium
  FurLength: Medium
  Vaccinated: Yes
  Sterilized: No
  Health: Healthy
  Description: Their pregnant mother was dumped by her irresponsible owner at the roadside near some shops in Subang Jaya. Gave birth to them at the roadside. They are all healthy and adorable puppies. Already dewormed, vaccinated and

## Choose what to mask

`manual_masks` says which fields to hide per row. These are the cells the models will be asked to recover. Only the listed columns (Gender, Color1, …) are eligible.

In [6]:
# --- Eval masks: programmatic, deterministic, swappable ---
# These are the cells we hide and ask the models to recover.
# Tweak EVAL_MASK_CONFIG and re-run downstream cells to try a different masking
# strategy without retraining.

EVAL_MASK_CONFIG = {
    "strategy": "random_count",          # "random_count" | "fixed_count" | "fixed_cols"
    "min_n": 1,
    "max_n": 3,
    "allowed_cols": [
        "Gender", "Color1", "Color2", "Color3",
        "MaturitySize", "FurLength",
        "Vaccinated", "Dewormed", "Sterilized",
        "Health", "Age",
    ],
    "fixed_cols": ["Gender", "Color1"],   # used only when strategy == "fixed_cols"
    "fixed_n": 2,                          # used only when strategy == "fixed_count"
    "seed": 0,
}

ELIGIBLE = set(EVAL_MASK_CONFIG["allowed_cols"])


def build_eval_masks(rows_df, config):
    rng = np.random.default_rng(config["seed"])
    masks = {}
    allowed = config["allowed_cols"]
    strategy = config["strategy"]
    for i in rows_df.index:
        if strategy == "random_count":
            n = int(rng.integers(config["min_n"], config["max_n"] + 1))
            cols = rng.choice(allowed, size=n, replace=False).tolist()
        elif strategy == "fixed_count":
            n = config["fixed_n"]
            cols = rng.choice(allowed, size=n, replace=False).tolist()
        elif strategy == "fixed_cols":
            cols = list(config["fixed_cols"])
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
        masks[i] = cols
    return masks


eval_masks = build_eval_masks(candidates_readable, EVAL_MASK_CONFIG)
total_cells = sum(len(v) for v in eval_masks.values())
per_col_counts = {}
for cols in eval_masks.values():
    for c in cols:
        per_col_counts[c] = per_col_counts.get(c, 0) + 1
print(f"Built eval_masks for {len(eval_masks)} rows, {total_cells} masked cells total")
print("Per-column counts:")
for c, n in sorted(per_col_counts.items(), key=lambda x: -x[1]):
    print(f"  {c:14s} {n}")


Built eval_masks for 200 rows, 383 masked cells total
Per-column counts:
  Dewormed       41
  FurLength      39
  Health         37
  Sterilized     37
  Gender         36
  Vaccinated     35
  Color3         35
  Age            35
  Color2         30
  Color1         29
  MaturitySize   29


## Render helpers

`render_for_llada` flattens the rows into one text blob, replacing each masked cell with `[MASK]` tokens sized to fit the original value. `find_mask_runs` later locates the contiguous mask spans in the tokenised input so predictions can be sliced back out per cell.

In [7]:
# --- Per-row rendering helpers (single-row, no header) ---
RENDER_FIELDS = ["Type","Name","Age","Breed1","Breed2","Gender",
                 "Color1","Color2","Color3","MaturitySize","FurLength",
                 "Vaccinated","Dewormed","Sterilized","Health","Description"]

MAX_DESC_WORDS = 80


def _format_value(field, value, max_desc_words=MAX_DESC_WORDS):
    val = str(value)
    if field == "Age":
        try:
            val = str(int(float(val)))
        except Exception:
            pass
    if field == "Description":
        words = val.split()
        if len(words) > max_desc_words:
            val = " ".join(words[:max_desc_words]) + "..."
    return val


def render_row_full(row, max_desc_words=MAX_DESC_WORDS):
    """No masks — used for training data + few-shot examples."""
    parts = []
    for f in RENDER_FIELDS:
        parts.append(f"{f}: {_format_value(f, row[f], max_desc_words)}\n")
    return "".join(parts)


def render_row_with_masks(row, masked_cols, tokenizer, max_desc_words=MAX_DESC_WORDS):
    """LLaDA-style: each masked value replaced with N mask tokens, sized to the
    original value's token length. Returns (text, [(col, original_value), ...])."""
    parts = []
    keys = []
    masked = set(masked_cols or [])
    mask_token = tokenizer.mask_token
    for f in RENDER_FIELDS:
        val = _format_value(f, row[f], max_desc_words)
        if f in masked:
            n_tok = max(1, len(tokenizer.encode(val, add_special_tokens=False)))
            parts.append(f"{f}: {mask_token * n_tok}\n")
            keys.append((f, val))
        else:
            parts.append(f"{f}: {val}\n")
    return "".join(parts), keys


def render_row_with_blanks(row, masked_cols, max_desc_words=MAX_DESC_WORDS, start_k=1):
    """Llama / chat-style: masked values replaced with [BLANK_k] placeholders.
    Returns (text, [(col, original_value), ...], next_k)."""
    parts = []
    keys = []
    masked = set(masked_cols or [])
    k = start_k
    for f in RENDER_FIELDS:
        val = _format_value(f, row[f], max_desc_words)
        if f in masked:
            parts.append(f"{f}: [BLANK_{k}]\n")
            keys.append((f, val))
            k += 1
        else:
            parts.append(f"{f}: {val}\n")
    return "".join(parts), keys, k


def find_mask_runs(input_ids, mask_id):
    """Return list of (start, end_exclusive) for each contiguous run of mask_id."""
    runs = []
    in_run = False
    s = 0
    for idx, t in enumerate(input_ids):
        if t == mask_id:
            if not in_run:
                s = idx
                in_run = True
        else:
            if in_run:
                runs.append((s, idx))
                in_run = False
    if in_run:
        runs.append((s, len(input_ids)))
    return runs


# --- Few-shot example pool for chat-style inference ---
# Sampled once, used identically for LLaDA-Instruct and Llama queries.
N_FEWSHOT = 2 if SMOKE_TEST else 5
FEWSHOT_SEED = 0

train_pool_for_fewshot = train.drop(candidate_train_indices).reset_index(drop=True)
train_pool_readable = to_readable(train_pool_for_fewshot)
_fs_rng = np.random.default_rng(FEWSHOT_SEED)
fewshot_indices = _fs_rng.choice(len(train_pool_readable), size=N_FEWSHOT, replace=False)
fewshot_rows = train_pool_readable.iloc[fewshot_indices].reset_index(drop=True)
fewshot_text = "\n".join(
    f"--- Example {j+1} ---\n{render_row_full(r)}"
    for j, r in fewshot_rows.iterrows()
)
print(f"Train pool: {len(train_pool_readable)} rows (200 candidates excluded)")
print(f"Sampled {N_FEWSHOT} few-shot example rows (seed={FEWSHOT_SEED})")
print(f"Few-shot block length: {len(fewshot_text)} chars")


Train pool: 14793 rows (200 candidates excluded)
Sampled 5 few-shot example rows (seed=0)
Few-shot block length: 2266 chars


## LLaDA-Instruct infill

Load the instruction-tuned 8B diffusion LM, mask all chosen cells in one big text, and run masked-diffusion sampling so every blank is filled in a single pass. Decode each mask run and store predictions keyed by `(row, column)`.

In [8]:
# --- LLaDA-Instruct: per-row chat-style infill with few-shot context ---
ll_model = dllm.utils.get_model(model_name_or_path="GSAI-ML/LLaDA-8B-Instruct").eval()
ll_tokenizer = dllm.utils.get_tokenizer(model_name_or_path="GSAI-ML/LLaDA-8B-Instruct")
sampler = MDLMSampler(model=ll_model, tokenizer=ll_tokenizer)
mask_id = ll_tokenizer.mask_token_id

INSTRUCTION_PREFIX = (
    "You are filling in missing values in a row from a pet adoption dataset. "
    "Below are example complete rows for reference, then a target row where "
    "some values have been replaced with [MASK] tokens. Fill in the [MASK] "
    "tokens with the most plausible values given the row's Description and "
    "the other fields.\n\n"
    "Examples:\n"
    f"{fewshot_text}\n\n"
    "--- Target row ---\n"
)

llada_preds = {}
saved_instruct = None  # for the step viewer
for i, row in candidates_readable.iterrows():
    masked_cols = eval_masks[i]
    if not masked_cols:
        continue
    masked_body, keys_for_row = render_row_with_masks(row, masked_cols, ll_tokenizer)
    user_content = INSTRUCTION_PREFIX + masked_body
    chat_text = ll_tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        add_generation_prompt=True, tokenize=False,
    )
    input_ids = ll_tokenizer.encode(chat_text, add_special_tokens=False)
    runs = find_mask_runs(input_ids, mask_id)
    if len(runs) != len(keys_for_row):
        print(f"Row {i}: mask runs {len(runs)} != expected {len(keys_for_row)}; skipping")
        continue
    n_masks = sum(e - s for s, e in runs)
    cfg = MDLMSamplerConfig(steps=n_masks, block_size=None, return_dict=True)
    out = sampler.infill([input_ids], cfg)
    seq = out.sequences[0].tolist()
    for (start, end), (col, orig) in zip(runs, keys_for_row):
        pred = ll_tokenizer.decode(seq[start:end]).strip()
        llada_preds[(i, col)] = pred
    if saved_instruct is None:
        saved_instruct = {
            "out": out, "runs": runs,
            "keys": [(i, c, o) for (c, o) in keys_for_row],
            "row_idx": i,
        }

print(f"\nLLaDA-Instruct predictions: {len(llada_preds)} cells across "
      f"{len({k[0] for k in llada_preds})} rows")
print("Sample predictions:")
for k in list(llada_preds.keys())[:8]:
    print(f"  Row {k[0]} {k[1]}: pred={llada_preds[k]!r}")

del ll_model, sampler
torch.cuda.empty_cache()
gc.collect()



LLaDA-Instruct predictions: 383 cells across 200 rows
Sample predictions:
  Row 0 Color2: pred='White'
  Row 0 FurLength: pred='Short'
  Row 0 Health: pred='Healthy'
  Row 1 Gender: pred='Male'
  Row 2 Sterilized: pred='No'
  Row 3 FurLength: pred='Long'
  Row 3 Health: pred='Healthy'
  Row 4 FurLength: pred='Medium'


306

## Step-through viewer (Instruct)

Build a tabular viewer over the saved denoising history. Cells turn green the step they get committed; still-masked cells show `_`; unmasked cells show their original known value.

In [9]:
# --- Sense-check: step through LLaDA-Instruct denoising for ONE row ---
# saved_instruct holds the diffusion history for the first row that was infilled.
import importlib, dllm.utils.visualizers, dllm.utils
importlib.reload(dllm.utils.visualizers)
importlib.reload(dllm.utils)
from dllm.utils import TableDiffViewer

_row_df = candidates_readable.loc[[saved_instruct["row_idx"]]]
tv_instruct = TableDiffViewer(
    saved_instruct["out"].histories,
    ll_tokenizer,
    saved_instruct["runs"],
    saved_instruct["keys"],
    rows_df=_row_df,
)


Advance one diffusion step.

In [10]:
tv_instruct.next()

Step 0/3  |  filled 0/3  |  committed this step: 0


,Color2,FurLength,Health
row,,,
0,_,_,_


## LLaDA-Base infill

Same task with the non-instruction-tuned base model — useful to see whether instruction tuning matters for tabular infill.

In [11]:
# --- LLaDA-8B-Base: per-row raw infill (vanilla + fine-tuned for comparison) ---
# Run inference twice with the same setup, swapping only the model weights.
# "vanilla" = the base model untouched; "ft" = base + your fine-tuned LoRA.
# This gives us the apples-to-apples comparison that tells us whether the
# fine-tune actually changed anything.

FT_CKPT = "/rds/general/user/rmw324/home/raels_playground/playground_1/dllm/.models/llada-base-petfinder-fast/checkpoint-final"
runs_to_do = [
    ("vanilla", "GSAI-ML/LLaDA-8B-Base"),
    ("ft",      FT_CKPT),
]

llada_base_preds_by_variant = {}
saved_base = None  # for the step viewer

for variant_name, model_path in runs_to_do:
    print(f"\n=== Running LLaDA-Base ({variant_name}): {model_path} ===")
    base_model = dllm.utils.get_model(model_name_or_path=model_path).eval()
    base_tokenizer = dllm.utils.get_tokenizer(model_name_or_path="GSAI-ML/LLaDA-8B-Base")
    base_sampler = MDLMSampler(model=base_model, tokenizer=base_tokenizer)
    base_mask_id = base_tokenizer.mask_token_id

    preds = {}
    for i, row in candidates_readable.iterrows():
        masked_cols = eval_masks[i]
        if not masked_cols:
            continue
        text, keys_for_row = render_row_with_masks(row, masked_cols, base_tokenizer)
        input_ids = base_tokenizer.encode(text, add_special_tokens=True)
        runs = find_mask_runs(input_ids, base_mask_id)
        if len(runs) != len(keys_for_row):
            print(f"  Row {i}: mask runs {len(runs)} != expected {len(keys_for_row)}; skipping")
            continue
        n_masks = sum(e - s for s, e in runs)
        cfg = MDLMSamplerConfig(steps=n_masks, block_size=None, return_dict=True)
        out = base_sampler.infill([input_ids], cfg)
        seq = out.sequences[0].tolist()
        for (start, end), (col, orig) in zip(runs, keys_for_row):
            preds[(i, col)] = base_tokenizer.decode(seq[start:end]).strip()
        if variant_name == "ft" and saved_base is None:
            saved_base = {
                "out": out, "runs": runs,
                "keys": [(i, c, o) for (c, o) in keys_for_row],
                "row_idx": i,
            }

    llada_base_preds_by_variant[variant_name] = preds
    print(f"  {variant_name}: {len(preds)} predictions across "
          f"{len({k[0] for k in preds})} rows")

    del base_model, base_sampler
    torch.cuda.empty_cache()
    gc.collect()

# Aliases consumed by the eval cell
llada_base_preds = llada_base_preds_by_variant["ft"]
llada_base_vanilla_preds = llada_base_preds_by_variant["vanilla"]

# --- Disagreement check: smoking-gun test that the LoRA actually applied ---
shared_keys = set(llada_base_preds) & set(llada_base_vanilla_preds)
disagree = sum(
    1 for k in shared_keys
    if llada_base_preds[k] != llada_base_vanilla_preds[k]
)
print(f"\nFT vs vanilla disagree on {disagree}/{len(shared_keys)} cells "
      f"({100*disagree/max(1,len(shared_keys)):.0f}%)")
if disagree == 0:
    print("  ⚠️  No differences — the LoRA did not change inference behaviour. "
          "Either the adapter did not load or 1 epoch produced negligible weight changes.")
else:
    print("  ✓ LoRA changed inference behaviour. Comparing accuracy in the eval cell "
          "will tell us if those changes were beneficial.")

print("\nSample predictions (FT):")
for k in list(llada_base_preds.keys())[:6]:
    print(f"  Row {k[0]} {k[1]}: ft={llada_base_preds[k]!r} vanilla={llada_base_vanilla_preds.get(k)!r}")



=== Running LLaDA-Base (vanilla): GSAI-ML/LLaDA-8B-Base ===
  vanilla: 383 predictions across 200 rows

=== Running LLaDA-Base (ft): /rds/general/user/rmw324/home/raels_playground/playground_1/dllm/.models/llada-base-petfinder-fast/checkpoint-final ===
  ft: 383 predictions across 200 rows

FT vs vanilla disagree on 244/383 cells (64%)
  ✓ LoRA changed inference behaviour. Comparing accuracy in the eval cell will tell us if those changes were beneficial.

Sample predictions (FT):
  Row 0 Color2: ft='White' vanilla='White'
  Row 0 FurLength: ft='Short' vanilla='Short'
  Row 0 Health: ft='Healthy' vanilla='Good'
  Row 1 Gender: ft='Mixed' vanilla='1'
  Row 2 Sterilized: ft='No' vanilla='No'
  Row 3 FurLength: ft='Long' vanilla='1'


## Step-through viewer (Base)

In [12]:
# --- Sense-check: step through LLaDA-Base denoising for ONE row ---
_row_df = candidates_readable.loc[[saved_base["row_idx"]]]
tv_base = TableDiffViewer(
    saved_base["out"].histories,
    base_tokenizer,
    saved_base["runs"],
    saved_base["keys"],
    rows_df=_row_df,
)
tv_base.next()


Step 0/3  |  filled 0/3  |  committed this step: 0


,Color2,FurLength,Health
row,,,
0,_,_,_


## Llama-3.1 baseline (autoregressive)

A standard left-to-right LM can't natively infill, so we replace masks with `[BLANK_k]` placeholders and ask Llama to output `[BLANK_k]: value` lines. Predictions are parsed back with a regex.

In [13]:
# --- Llama-3.1-8B-Instruct: per-row chat-style fill with few-shot examples ---
from transformers import AutoModelForCausalLM, AutoTokenizer

LLAMA = "meta-llama/Llama-3.1-8B-Instruct"
llama_tok = AutoTokenizer.from_pretrained(LLAMA)
llama = AutoModelForCausalLM.from_pretrained(
    LLAMA, torch_dtype=torch.bfloat16, device_map="cuda"
).eval()

LLAMA_INSTRUCTION_PREFIX = (
    "You are filling in missing values in a row from a pet adoption dataset. "
    "Below are example complete rows for reference, then a target row where "
    "some values are labelled [BLANK_1], [BLANK_2], etc.\n\n"
    "Examples:\n"
    f"{fewshot_text}\n\n"
    "--- Target row ---\n"
)

LLAMA_RESPONSE_HINT = (
    "\nOutput exactly one line per blank in this format:\n"
    "[BLANK_1]: <value>\n[BLANK_2]: <value>\n...\n"
    "Use only short answers like 'Male', 'Medium', 'Yes', 'Black', '24'. "
    "Do not include any explanation."
)

llama_preds = {}
pat = re.compile(r"\[BLANK_(\d+)\]\s*:\s*(.+)")
for i, row in candidates_readable.iterrows():
    masked_cols = eval_masks[i]
    if not masked_cols:
        continue
    body, keys_for_row, _ = render_row_with_blanks(row, masked_cols, start_k=1)
    user_content = LLAMA_INSTRUCTION_PREFIX + body + LLAMA_RESPONSE_HINT
    messages = [{"role": "user", "content": user_content}]
    inp = llama_tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        gen = llama.generate(inp, max_new_tokens=20 * len(keys_for_row), do_sample=False)
    response = llama_tok.decode(gen[0, inp.shape[1]:], skip_special_tokens=True)
    # Parse [BLANK_k]: value lines
    parsed = {}
    for line in response.splitlines():
        m = pat.search(line)
        if m:
            k = int(m.group(1))
            if 1 <= k <= len(keys_for_row):
                parsed[k] = m.group(2).strip().rstrip(".,")
    for k_idx, (col, orig) in enumerate(keys_for_row, start=1):
        llama_preds[(i, col)] = parsed.get(k_idx, "")

n_parsed = sum(1 for v in llama_preds.values() if v)
print(f"Llama parsed {n_parsed}/{len(llama_preds)} blanks across "
      f"{len({k[0] for k in llama_preds})} rows")

del llama, llama_tok
torch.cuda.empty_cache()
gc.collect()


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, yo

Llama parsed 383/383 blanks across 200 rows


0

## TabPFN baseline (purely tabular)

A non-LM control: train a TabPFN classifier (or regressor for Age) per masked column using the other columns as features. Sees zero of the `Description` text — purely a tabular prior over the dataset.

In [14]:

from tabpfn import TabPFNClassifier, TabPFNRegressor

TABPFN_FEATURES = ["Type","Age","Breed1","Breed2","Gender","Color1","Color2","Color3",
                   "MaturitySize","FurLength","Vaccinated","Dewormed","Sterilized",
                   "Health","Quantity","Fee","State"]

DECODE_FOR_COL = {
    "Gender": GENDER_MAP, "MaturitySize": SIZE_MAP, "FurLength": FUR_MAP,
    "Vaccinated": YESNO_MAP, "Sterilized": YESNO_MAP, "Dewormed": YESNO_MAP,
    "Health": HEALTH_MAP,
    "Color1": color_map, "Color2": color_map, "Color3": color_map,
}

# Group masked cells by column
by_col = {}
for row_i, cols in eval_masks.items():
    for c in cols:
        by_col.setdefault(c, []).append(row_i)

# Exclude the candidate rows from TabPFN's in-context training pool
train_for_tabpfn = train.drop(candidate_train_indices)
fit_pool = train_for_tabpfn.sample(
    n=min(8000, len(train_for_tabpfn)), random_state=0
).reset_index(drop=True)

tabpfn_preds = {}
for col, row_indices in by_col.items():
    feats = [c for c in TABPFN_FEATURES if c != col]
    X_train = fit_pool[feats].values
    y_train = fit_pool[col].values
    X_test  = candidates.iloc[row_indices][feats].values

    if col == "Age":
        m = TabPFNRegressor()
    else:
        m = TabPFNClassifier()
    m.fit(X_train, y_train)
    preds = m.predict(X_test)

    for i, row_i in enumerate(row_indices):
        p = preds[i]
        if col == "Age":
            tabpfn_preds[(row_i, col)] = str(int(round(float(p))))
        else:
            decoder = DECODE_FOR_COL.get(col, {})
            tabpfn_preds[(row_i, col)] = decoder.get(int(p), str(p))

print(f"TabPFN predictions: {len(tabpfn_preds)}")
for k, v in list(tabpfn_preds.items())[:8]:
    print(f"  {k}: {v!r}")


TabPFN predictions: 383
  (0, 'Health'): 'Healthy'
  (3, 'Health'): 'Healthy'
  (4, 'Health'): 'Healthy'
  (6, 'Health'): 'Healthy'
  (17, 'Health'): 'Healthy'
  (22, 'Health'): 'Healthy'
  (27, 'Health'): 'Healthy'
  (28, 'Health'): 'Healthy'


## Evaluation

Per-column accuracy for categorical fields, MAE for Age. Plus a few worked rows so you can eyeball where each method agrees or disagrees with the ground truth.

In [15]:
# --- Evaluation: accuracy for categorical, MAE for Age ---
def norm(s): return str(s).strip().lower().rstrip(".,")

# Ground truth (readable strings, original Age)
gt = {}
for row_i, cols in eval_masks.items():
    for c in cols:
        if c == "Age":
            gt[(row_i, c)] = str(int(float(candidates.loc[row_i, "Age"])))
        else:
            gt[(row_i, c)] = str(candidates_readable.loc[row_i, c])

models = {
    "LLaDA-Instruct":       llada_preds,
    "LLaDA-Base (vanilla)": llada_base_vanilla_preds,
    "LLaDA-Base (FT)":      llada_base_preds,
    "Llama":                llama_preds,
    "TabPFN":               tabpfn_preds,
}
all_cols = sorted({c for cols in eval_masks.values() for c in cols})

rows_out = {}
for name, preds in models.items():
    rows_out[name] = {}
    for c in all_cols:
        ks = [k for k in gt if k[1] == c]
        if c == "Age":
            errs = []
            for k in ks:
                try: errs.append(abs(float(preds.get(k, "")) - float(gt[k])))
                except: errs.append(np.nan)
            rows_out[name][c] = f"MAE={np.nanmean(errs):.1f}" if errs else "—"
        else:
            hits = sum(norm(preds.get(k, "")) == norm(gt[k]) for k in ks)
            rows_out[name][c] = f"{hits}/{len(ks)} ({hits/len(ks):.0%})" if ks else "—"

results_df = pd.DataFrame(rows_out).T[all_cols]
print("=== Results (per-field per-model) ===")
print(results_df.to_string())

print("\n=== Worked examples ===")
for k in list(gt.keys())[:6]:
    row_i, col = k
    print(f"\nRow {row_i+1}  {col}:  truth={gt[k]!r}")
    for name in models:
        print(f"    {name:15s} {models[name].get(k, '')!r}")


=== Results (per-field per-model) ===
                           Age       Color1       Color2       Color3     Dewormed    FurLength       Gender        Health MaturitySize   Sterilized   Vaccinated
LLaDA-Instruct         MAE=9.7  19/29 (66%)   6/30 (20%)  26/35 (74%)  35/41 (85%)  30/39 (77%)  28/36 (78%)  37/37 (100%)  21/29 (72%)  32/37 (86%)  19/35 (54%)
LLaDA-Base (vanilla)   MAE=4.7  14/29 (48%)  11/30 (37%)  14/35 (40%)  11/41 (27%)  18/39 (46%)  24/36 (67%)    5/37 (14%)    2/29 (7%)  15/37 (41%)  10/35 (29%)
LLaDA-Base (FT)        MAE=9.7  13/29 (45%)   8/30 (27%)  18/35 (51%)  26/41 (63%)  28/39 (72%)  31/36 (86%)  37/37 (100%)  13/29 (45%)  25/37 (68%)  14/35 (40%)
Llama                 MAE=12.6  12/29 (41%)   9/30 (30%)  22/35 (63%)  21/41 (51%)  25/39 (64%)  23/36 (64%)  37/37 (100%)  18/29 (62%)  26/37 (70%)   7/35 (20%)
TabPFN                 MAE=8.5  15/29 (52%)   9/30 (30%)   4/35 (11%)  31/41 (76%)  30/39 (77%)  18/36 (50%)  37/37 (100%)  18/29 (62%)  30/37 (81%)  28

## Fine-tune LLaDA on PetFinder (data prep)

Build a pre-tokenised `DatasetDict` of full PetFinder rows (no masks). The MDLM trainer dynamically masks tokens at training time, so we only need to provide the full table-row text. The 25 candidate test rows are excluded to avoid leakage.

In [16]:
# --- Build the SFT training dataset for LLaDA-Base ---
# Single-row template (no === Row N === header), 80-word description cap,
# tokenised with the LLaDA-Base tokenizer, candidates excluded.
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

OUT_DIR = "/rds/general/user/rmw324/home/raels_playground/datasets/petfinder_llada_sft_v2"
if SMOKE_TEST:
    OUT_DIR = OUT_DIR + "_smoke"

# 1. Exclude the candidate rows (already computed in cell 6)
train_pool = train.drop(candidate_train_indices).reset_index(drop=True)
if SMOKE_TEST:
    train_pool = train_pool.sample(n=500, random_state=0).reset_index(drop=True)
    print(f"  SMOKE_TEST: subsampled train_pool to {len(train_pool)} rows")
train_pool_readable = to_readable(train_pool)
print(f"Training pool: {len(train_pool_readable)} rows "
      f"(excluding {len(candidate_train_indices)} candidates)")

# 2. Tokenise each row with the LLaDA-Base tokenizer; labels = input_ids (full MDLM loss)
tok = AutoTokenizer.from_pretrained("GSAI-ML/LLaDA-8B-Base")

records = []
for _, row in train_pool_readable.iterrows():
    text = render_row_full(row, max_desc_words=MAX_DESC_WORDS)
    ids = tok.encode(text, add_special_tokens=True)
    records.append({"input_ids": ids, "labels": list(ids)})

ds = Dataset.from_list(records)
lens = [len(r["input_ids"]) for r in records]
print(f"Tokenised. Mean length: {sum(lens)/len(lens):.1f} tokens, max: {max(lens)}")
print(f"  P95: {sorted(lens)[int(0.95*len(lens))]}, P99: {sorted(lens)[int(0.99*len(lens))]}")

# 3. ~10% internal val split (capped at 1500 for stability)
if SMOKE_TEST:
    n_val = 50
else:
    n_val = min(1500, max(200, len(records) // 10))
split = ds.train_test_split(test_size=n_val, seed=0)
dsd = DatasetDict({"train": split["train"], "test": split["test"]})
dsd.save_to_disk(OUT_DIR)
print(f"Saved {len(dsd['train'])} train + {len(dsd['test'])} test → {OUT_DIR}")


Training pool: 14793 rows (excluding 200 candidates)
Tokenised. Mean length: 144.6 tokens, max: 654
  P95: 196, P99: 236


Saving the dataset (0/1 shards):   0%|          | 0/13314 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1479 [00:00<?, ? examples/s]

Saved 13314 train + 1479 test → /rds/general/user/rmw324/home/raels_playground/datasets/petfinder_llada_sft_v2


In [17]:
from datasets import load_from_disk

dsd = load_from_disk(OUT_DIR)
print(dsd)

# Decode 3 random rows back to text
for i in range(3):
    ids = dsd["train"][i]["input_ids"]
    print(f"\n--- Sample {i} (len={len(ids)}) ---")
    print(tok.decode(ids))


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 13314
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 1479
    })
})

--- Sample 0 (len=141) ---
Type: Dog
Name: Roku
Age: 3
Breed1: Mixed Breed
Breed2: None
Gender: Female
Color1: Black
Color2: Brown
Color3: White
MaturitySize: Medium
FurLength: Short
Vaccinated: Yes
Dewormed: Yes
Sterilized: No
Health: Healthy
Description: We found this puppy by the roadside with the other puppy. Due to financial problem we could only adopt 1. She is quiet and tend to stick with people. Well-behaved but may expect some nibbling as she is still teething. She just had her vaccination and dewormed.


--- Sample 1 (len=105) ---
Type: Dog
Name: Mr. Xiao Hei
Age: 3
Breed1: Mixed Breed
Breed2: Mixed Breed
Gender: Male
Color1: Black
Color2: None
Color3: None
MaturitySize: Medium
FurLength: Short
Vaccinated: No
Dewormed: Yes
Sterilized: No
Health: Healthy
Description: Puppies

## Run training (terminal command)

Run this in a terminal, **not** the notebook (training an 8B model in a Jupyter cell will exhaust the kernel and you can't watch it). bf16 + LoRA, single RTX 6000:

```bash
cd /rds/general/user/rmw324/home/raels_playground/playground_1/dllm

accelerate launch \
    --config_file scripts/accelerate_configs/ddp.yaml --num_processes 1 \
    examples/llada/sft.py \
    --model_name_or_path GSAI-ML/LLaDA-8B-Base \
    --dataset_args /rds/general/user/rmw324/home/raels_playground/datasets/petfinder_llada_sft_v2 \
    --load_preprocessed_data True \
    --lora True \
    --output_dir .models/llada-base-petfinder \
    --num_train_epochs 3 \
    --learning_rate 2e-4 \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --max_length 256 \
    --bf16 True \
    --logging_steps 25 \
    --save_strategy epoch
```

Notes:
- Fine-tuning **LLaDA-Base** (not Instruct) so the raw infill template at inference matches the training format byte-for-byte.
- LoRA fine-tunes ~10M params (frozen base, ~16 GB weights in bf16). Should sit ~20–25 GB VRAM.
- Effective batch = 2 × 8 = 16. With ~14k rows × 3 epochs ≈ 2,600 optimiser steps. Estimate ~30–60 min on RTX 6000.
- Check the printed P95/P99 lengths from cell 30. If P99 exceeds 256 with the 80-word desc cap, bump `--max_length` to 384.
- Output checkpoint goes to `.models/llada-base-petfinder/checkpoint-final/`. Load it back via `dllm.utils.get_model(model_name_or_path=".models/llada-base-petfinder/checkpoint-final")` and rerun the LLaDA-Base inference + evaluation cells with that model.


In [18]:
!nvidia-smi

Mon May 11 23:42:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:CA:00.0 Off |                    0 |
|  0%   54C    P0             84W /  300W |    1397MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----